# SIMBES — Módulo 5: Sensores y Monitoreo BES

**Objetivo:** Interpretar las señales de los sensores operativos de un BES:
1. Cartas amperimétricas — patrones de corriente y diagnóstico
2. Temperatura de fondo — gradiente geotérmico y Arrhenius
3. Vibración — umbrales ISO 10816-3 y diagnóstico de causa
4. Diagnóstico integrado — correlación multi-sensor

---
**Fuentes:** API RP 11S5 · ISO 10816-3 · CLAUDE.md §2.6  
**Unidades:** mixtas campo (A, mm/s, psi, °C, m)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'backend'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.rcParams.update({
    'figure.facecolor': '#0B0F1A', 'axes.facecolor': '#111827',
    'axes.edgecolor': '#1E293B', 'axes.labelcolor': '#CBD5E1',
    'xtick.color': '#64748B', 'ytick.color': '#64748B',
    'text.color': '#CBD5E1', 'grid.color': '#1E293B',
    'grid.linestyle': '--', 'legend.facecolor': '#111827',
    'legend.edgecolor': '#1E293B', 'font.family': 'monospace',
})

# ── Physics Python (inline — sensors.py aún no existe como módulo backend) ──

def bottomhole_temperature(T_surface_C, depth_m, gradient_C_100m=3.0):
    """T_BH = T_surface + gradient × depth"""
    T_BH = T_surface_C + (gradient_C_100m / 100) * depth_m
    return {'T_bottomhole_C': round(T_BH, 1), 'T_cable_avg_C': round((T_surface_C + T_BH)/2, 1)}

def arrhenius_life_pct(T_op, T_rated):
    dT = T_op - T_rated
    if dT <= 0: return 100.0
    return round(2**(-dT/10) * 100, 1)

def vibration_rms(signal):
    return np.sqrt(np.mean(np.array(signal)**2))

print('Setup OK ✓')

## 1. Cartas Amperimétricas — Patrones de Corriente

La corriente del motor BES es el sensor más universal. Su patrón temporal revela la condición operativa:

| Patrón | Descripción | Causa | Acción |
|--------|-------------|-------|--------|
| Estable ≈100% | Normal | BEP óptimo | Ninguna |
| Oscilante ±15–25% | Surging | Fuera del BEP | Ajustar VSD |
| Baja <70% sostenida | Subcarga | Gas/eje roto | Investigar |
| Alta >115% sostenida | Sobrecarga | Viscosidad/sólidos | Revisar fluido |
| Caída súbita <20% | Gas Lock | Gas en bomba | **PARAR** |

In [ ]:
# Generación de señales sintéticas de corriente (determinísticas)
np.random.seed(42)
I_rated = 80  # A
t = np.linspace(0, 60, 120)  # 60 segundos, 120 puntos

def make_current(condition, t, I_rated=80):
    np.random.seed(42)
    noise = np.random.randn(len(t))
    if condition == 'normal':
        return I_rated * (1.0 + 0.02 * noise)
    elif condition == 'surging':
        return I_rated * (1.0 + 0.18 * np.sin(2*np.pi*0.4*t) + 0.04 * noise)
    elif condition == 'underload':
        return I_rated * (0.62 + 0.03 * noise)
    elif condition == 'overload':
        return I_rated * (1.25 + 0.05 * noise)
    elif condition == 'gas_lock':
        I = np.where(t < 20,
            I_rated * (1.0 + 0.02 * noise),
            np.where(t < 26,
                I_rated * (1.0 - (t - 20)/6 * 0.82 + 0.03 * noise),
                I_rated * (0.12 + 0.08 * np.abs(np.sin(2*np.pi*0.15*t)) + 0.04*np.abs(noise))
            ))
        return np.maximum(0, I)

CONDITIONS = [
    ('normal',    '#22C55E', 'Normal'),
    ('surging',   '#F59E0B', 'Surging'),
    ('underload', '#60A5FA', 'Subcarga'),
    ('overload',  '#FB923C', 'Sobrecarga'),
    ('gas_lock',  '#EF4444', 'Gas Lock'),
]

fig, axes = plt.subplots(5, 1, figsize=(12, 12), sharex=True)
for ax, (cond, color, label) in zip(axes, CONDITIONS):
    I = make_current(cond, t, I_rated)
    ax.plot(t, I, color=color, lw=1.8)
    ax.axhline(I_rated,       color='#64748B', lw=1, ls='--', alpha=0.7)
    ax.axhline(I_rated*1.15,  color='#F59E0B', lw=0.8, ls=':', alpha=0.6)
    ax.axhline(I_rated*0.70,  color='#60A5FA', lw=0.8, ls=':', alpha=0.6)
    ax.set_ylabel(label, color=color, fontsize=9)
    ax.set_ylim(0, I_rated * 1.5)
    ax.grid(True, alpha=0.3)
    avg = np.mean(I)
    ax.text(58, I_rated*1.35, f'avg={avg:.0f}A ({avg/I_rated*100:.0f}%)', 
            ha='right', fontsize=8, color=color)

axes[-1].set_xlabel('Tiempo [s]')
plt.suptitle(f'Cartas Amperimétricas — Patrones característicos (I_rated={I_rated} A)',
             color='#CBD5E1', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

## 2. Temperatura de Fondo — Gradiente Geotérmico

$$
T_{\text{BH}} = T_{\text{superficie}} + \frac{\text{gradiente}}{100} \times \text{profundidad}
$$

La temperatura en el fondo determina:
- La clase de aislamiento requerida (F/H/C)
- La corrección de resistencia del cable (Módulo 4)
- El riesgo Arrhenius si supera el límite nominal

In [ ]:
depths   = np.linspace(500, 5000, 200)
T_surf   = 25  # °C
gradients = [(2.5, '#22C55E', 'Grad. bajo (2.5°C/100m)'),
             (3.0, '#FBBF24', 'Grad. normal (3.0°C/100m)'),
             (4.0, '#F59E0B', 'Grad. alto (4.0°C/100m)'),
             (5.0, '#EF4444', 'Grad. muy alto (5.0°C/100m)')]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel izquierdo: T_BH vs profundidad
ax = axes[0]
for grad, color, label in gradients:
    T_BH = T_surf + (grad/100) * depths
    ax.plot(depths, T_BH, color=color, lw=2, label=label)

for T_class, name, color in [(155, 'Clase F', '#60A5FA'), (180, 'Clase H', '#A78BFA'), (220, 'Clase C', '#34D399')]:
    ax.axhline(T_class, color=color, lw=1, ls='--', alpha=0.6, label=f'{name} ({T_class}°C)')

ax.set_xlabel('Profundidad [m]')
ax.set_ylabel('T° fondo [°C]')
ax.set_title(f'T° de fondo vs. profundidad (T_sup={T_surf}°C)', pad=10)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.4)

# Panel derecho: vida Arrhenius vs T_BH para clase H
ax2 = axes[1]
T_BH_range = np.linspace(140, 230, 200)
for T_rated, label, color in [(155, 'Clase F', '#60A5FA'), (180, 'Clase H', '#A78BFA'), (220, 'Clase C', '#34D399')]:
    life = [arrhenius_life_pct(T, T_rated) for T in T_BH_range]
    ax2.plot(T_BH_range, life, color=color, lw=2, label=label)
    ax2.axvline(T_rated, color=color, lw=0.8, ls=':', alpha=0.5)

ax2.axhline(50, color='#F59E0B', lw=1, ls='--', alpha=0.6, label='50% vida')
ax2.set_xlabel('T° motor [°C]')
ax2.set_ylabel('Vida útil aislamiento [%]')
ax2.set_title('Vida Arrhenius por clase de aislamiento', pad=10)
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.4)
ax2.set_ylim(0, 105)

plt.suptitle('Temperatura Downhole — Gradiente Geotérmico y Arrhenius', color='#CBD5E1', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

## 3. Vibración — Señales y Diagnóstico

### Zonas operativas (ISO 10816-3 / API RP 11S5)

| Zona | RMS [mm/s] | Estado | Acción |
|------|-----------|--------|--------|
| A | < 4.0 | Normal | Registrar como baseline |
| B | 4.0–6.3 | Alerta temprana | Investigar causa |
| C | > 6.3 | Crítico | Paro recomendado |

In [ ]:
dt     = 0.005  # 5ms → 200 Hz
t_vib  = np.arange(0, 0.5, dt)  # 0.5 segundos
f_rot  = 60.0  # Hz (3600 RPM)

np.random.seed(77)

def make_vibration(condition, t, f_rot=60.0):
    np.random.seed(77)
    noise = np.random.randn(len(t))
    if condition == 'normal':
        return 0.8*noise + 0.3*np.sin(2*np.pi*f_rot*t)
    elif condition == 'desbalanceo':
        return 4.5*np.sin(2*np.pi*f_rot*t) + 1.0*np.sin(2*np.pi*2*f_rot*t) + 0.6*noise
    elif condition == 'rodamiento':
        f_bpfo = 5 * f_rot
        impulses = np.where(np.sin(2*np.pi*f_bpfo*t) > 0.85, 7.5*np.abs(noise), 0)
        return 0.5*noise + impulses
    elif condition == 'cavitacion':
        return 3.5*noise + 2.0*np.sin(2*np.pi*80*t)*np.abs(noise)

VIBE_CONDITIONS = [
    ('normal',      '#22C55E', 'Normal'),
    ('desbalanceo', '#F59E0B', 'Desbalanceo'),
    ('rodamiento',  '#FB923C', 'Rodamiento (BPFO)'),
    ('cavitacion',  '#EF4444', 'Cavitación'),
]

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
for ax, (cond, color, label) in zip(axes, VIBE_CONDITIONS):
    v = make_vibration(cond, t_vib, f_rot)
    rms = vibration_rms(v)
    zone = 'A (OK)' if rms < 4.0 else 'B (Alerta)' if rms < 6.3 else 'C (Crítico)'
    zone_color = '#22C55E' if rms < 4.0 else '#F59E0B' if rms < 6.3 else '#EF4444'
    ax.plot(t_vib, v, color=color, lw=1.2, alpha=0.9)
    ax.axhline(0, color='#1E293B', lw=0.5)
    ax.set_ylabel(label, color=color, fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.text(0.48, ax.get_ylim()[1]*0.85,
            f'RMS={rms:.1f} mm/s → Zona {zone}',
            ha='right', fontsize=8, color=zone_color)

axes[-1].set_xlabel('Tiempo [s]')
plt.suptitle('Señales de Vibración — Patrones y Diagnóstico (3600 RPM)',
             color='#CBD5E1', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

## 4. Diagnóstico Integrado — Caso Pozo Cóndor-8

In [ ]:
# ── Escenario Cóndor-8 ───────────────────────────────────────────
DEPTH_M     = 2200
T_SURF      = 25
GRADIENT    = 4.5   # °C/100m — pozo caliente
PS_PSI      = 650   # psi — baja presión de succión
PB_PSI      = 1800  # psi — presión de burbuja alta
T_RATED     = 180   # Clase H
I_RATED     = 95    # A
VIBE_RMS    = 5.8   # mm/s — zona B

pt = bottomhole_temperature(T_SURF, DEPTH_M, GRADIENT)
T_motor = pt['T_bottomhole_C']
life_pct = arrhenius_life_pct(T_motor, T_RATED)
ps_pb_ratio = PS_PSI / PB_PSI

print("POZO CÓNDOR-8 — Diagnóstico Integrado")
print("=" * 50)
print(f"  T° fondo      : {T_motor}°C (límite clase H = {T_RATED}°C)")
print(f"  ΔT Arrhenius  : {T_motor - T_RATED:.0f}°C → vida = {life_pct:.0f}%")
print(f"  Ps / Pb       : {PS_PSI} / {PB_PSI} psi = {ps_pb_ratio*100:.0f}%")
print(f"  Vibración RMS : {VIBE_RMS} mm/s (Zona B)")
print()
print("DIAGNÓSTICOS:")
if PS_PSI < PB_PSI:
    sev = 'CRÍTICO' if PS_PSI < PB_PSI * 0.5 else 'ADVERTENCIA'
    print(f"  [{sev}] Gas libre en succión — Ps < Pb. Instalar separador.")
if T_motor > T_RATED:
    print(f"  [ADVERTENCIA] T° motor excede límite clase H. Vida = {life_pct:.0f}%.")
if VIBE_RMS > 4.0:
    zone = 'CRÍTICO' if VIBE_RMS > 6.3 else 'ADVERTENCIA'
    print(f"  [{zone}] Vibración {VIBE_RMS} mm/s — Zona B. Investigar rodamiento.")

In [ ]:
# ── Dashboard visual del caso ─────────────────────────────────────
t_curr = np.linspace(0, 60, 120)
I_surging = make_current('surging', t_curr, I_RATED)
v_rodamiento = make_vibration('rodamiento', t_vib, f_rot)

fig = plt.figure(figsize=(13, 9))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.3)

# Carta amperimérica
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(t_curr, I_surging, color='#F59E0B', lw=1.8)
ax1.axhline(I_RATED, color='#64748B', lw=1, ls='--', label=f'I_rated={I_RATED}A')
ax1.axhline(I_RATED*1.15, color='#FB923C', lw=0.8, ls=':', alpha=0.7)
ax1.axhline(I_RATED*0.70, color='#60A5FA', lw=0.8, ls=':', alpha=0.7)
ax1.set_title('Carta Amperimérica — Surging detectado', fontsize=10)
ax1.set_xlabel('Tiempo [s]')
ax1.set_ylabel('Corriente [A]')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# Vibración
ax2 = fig.add_subplot(gs[1, :])
ax2.plot(t_vib, v_rodamiento, color='#FB923C', lw=1.3)
rms_actual = vibration_rms(v_rodamiento)
ax2.set_title(f'Vibración — Patrón rodamiento · RMS={rms_actual:.1f} mm/s (Zona B)', fontsize=10)
ax2.set_xlabel('Tiempo [s]')
ax2.set_ylabel('[mm/s]')
ax2.grid(True, alpha=0.3)

# T° downhole
ax3 = fig.add_subplot(gs[2, 0])
depths_plot = np.linspace(0, DEPTH_M, 100)
T_plot = T_SURF + (GRADIENT/100) * depths_plot
ax3.plot(T_plot, depths_plot, color='#FBBF24', lw=2)
ax3.axvline(T_motor, color='#EF4444', lw=1.5, ls='--', label=f'T_motor={T_motor}°C')
ax3.axvline(T_RATED, color='#22C55E', lw=1.2, ls=':', label=f'Límite H ({T_RATED}°C)')
ax3.scatter([T_motor], [DEPTH_M], color='#EF4444', s=80, zorder=5)
ax3.set_xlabel('Temperatura [°C]')
ax3.set_ylabel('Profundidad [m]')
ax3.invert_yaxis()
ax3.set_title(f'Perfil T° — grad={GRADIENT}°C/100m', fontsize=10)
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.3)

# Dashboard texto
ax4 = fig.add_subplot(gs[2, 1])
ax4.axis('off')
summary = (
    f"CÓNDOR-8 — RESUMEN SENSORES\n"
    f"{'='*30}\n\n"
    f"Corriente   : Surging (±18%)\n"
    f"Vibración   : {VIBE_RMS} mm/s (Zona B)\n"
    f"T° motor    : {T_motor}°C (ΔT={T_motor-T_RATED:.0f}°C)\n"
    f"Vida aislam.: {life_pct:.0f}% del nominal\n"
    f"Ps/Pb       : {ps_pb_ratio*100:.0f}% — gas libre\n"
    f"\nACCIONES PRIORITARIAS:\n"
    f"1. Ajustar VSD para eliminar surging\n"
    f"2. Instalar AGS (Ps << Pb)\n"
    f"3. Planificar extracción (rodamiento)\n"
    f"4. Monitorear T° motor diariamente\n"
)
ax4.text(0.05, 0.95, summary, transform=ax4.transAxes,
         fontsize=9, va='top', ha='left', color='#CBD5E1',
         fontfamily='monospace', linespacing=1.6)

plt.suptitle('Pozo Cóndor-8 — Dashboard Integrado de Sensores',
             color='#CBD5E1', fontsize=11, y=1.01)
plt.show()

## 5. Resumen — Reglas Operativas M5

| Sensor | Umbral | Diagnóstico | Acción |
|--------|--------|-------------|--------|
| Corriente oscilante ±15% | 0.3–0.8 Hz | Surging | Ajustar VSD / Ps |
| Corriente <70% sostenida | — | Subcarga / Gas | Verificar GVF |
| Corriente <20% súbita | — | Gas Lock | **PARAR** |
| Vibración RMS | > 4 mm/s | Zona B — alerta | Investigar causa |
| Vibración RMS | > 6.3 mm/s | Zona C — crítico | Paro recomendado |
| Ps/Pb | < 100% | Gas libre | Evaluar AGS |
| Ps/Pb | < 50% | Gas crítico | AGS urgente |
| T_motor > T_rated | ΔT = 10°C | Vida = 50% | Revisar refrigeración |

---
*Notebook generado automáticamente por SIMBES para el Módulo 5 — Sensores y Monitoreo.*  
*Versión: 1.0 · Fecha: 2026-03-13*